# CoRAL-Sep Evaluation: 8-Condition × 4-N Matrix

Runs the full evaluation matrix and produces:
- `results/eval_matrix.jsonl` — per-mixture records
- `results/summary.csv` — aggregated by (condition, N)
- `results/bootstrap_ci.json` — 95% BCa confidence intervals

Conditions: clean, reverb, noise, codec, reverb+noise, reverb+codec,
noise+codec, reverb+noise+codec.
N ∈ {2, 3, 4, 5}.

In [ ]:
CHECKPOINT_DIR = "/kaggle/working/joint"
EVAL_MANIFEST  = "data/eval/fixed_eval_manifest.jsonl"
OUTPUT_DIR     = "/kaggle/working/results"
DNSMOS_MODEL   = None   # path to sig_bak_ovrl.onnx, or None
DEVICE         = "cuda"
REPO_PATH      = "/kaggle/input/calmsep-code"
N_PER_BUCKET   = 50     # mixtures per (condition, N) bucket

In [ ]:
import os
import sys

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Build pipeline ────────────────────────────────────────────────────────────
from models.experts.srcorrnet import SRCorrNetExpert
from pipeline.infer import CoralSepPipeline, InferenceCfg

expert = SRCorrNetExpert(device=DEVICE)
cfg = InferenceCfg(device=DEVICE)
pipeline = CoralSepPipeline(expert=expert, cfg=cfg)

# Load trained components if available.
# pipeline.lora, pipeline.gate, etc. can be loaded here.
print("Pipeline ready.")

In [ ]:
# ── DNSMOS scorer ─────────────────────────────────────────────────────────────
from eval.dnsmos import DnsmosScorer

dnsmos = DnsmosScorer(model_path=DNSMOS_MODEL)
print(f"DNSMOS available: {dnsmos.is_available}")

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────
from eval.matrix import run_eval_matrix

matrix = run_eval_matrix(
    pipeline=pipeline,
    eval_manifest=EVAL_MANIFEST,
    dnsmos_scorer=dnsmos,
    max_per_bucket=N_PER_BUCKET,
    device=DEVICE,
)

print(f"Evaluated {len(matrix.records)} mixtures.")
matrix.to_jsonl(os.path.join(OUTPUT_DIR, "eval_matrix.jsonl"))
matrix.to_summary_csv(os.path.join(OUTPUT_DIR, "summary.csv"))
print("Saved results.")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
from eval.stats import format_summary_table

summary = matrix.summary_by_condition()
print(format_summary_table(summary, metric="si_sdri_mean", caption="SI-SDRi (dB)"))
print()
print(format_summary_table(summary, metric="count_acc", caption="Count Accuracy"))

In [ ]:
# ── Bootstrap CIs ─────────────────────────────────────────────────────────────
import json
from collections import defaultdict

from eval.stats import compute_ci_table

buckets = defaultdict(list)
for rec in matrix.records:
    buckets[(rec.condition, rec.n_true)].append(rec.si_sdri)

ci_table = compute_ci_table(dict(buckets), n_boot=2000)

# Convert tuple keys to strings for JSON serialization.
ci_json = {f"{k[0]}__N{k[1]}": v for k, v in ci_table.items()}
ci_path = os.path.join(OUTPUT_DIR, "bootstrap_ci.json")
with open(ci_path, "w") as f:
    json.dump(ci_json, f, indent=2)
print(f"Bootstrap CIs saved to {ci_path}")